# 00 - Optional Combined Phase 1 Research Run

Optional one-session workflow for historical training reuse, recency research, diagnostics, and bundle generation. Prefer notebooks 04 and 05 separately when GPU quota matters.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime -> T4/A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
REPORT_DIR  = f'{DRIVE_BASE}/reports'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'research/next-candidate-v1')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
research = cfg.get('experiments', {}).get('next_research_cycle', {})
assert research.get('same_window_selection_allowed') is False
assert research.get('new_future_oos_anchor_required') is True
print('Policy review status:', cfg.get('experiments', {}).get('policy_review', {}).get('status'))
print('Research cycle:', research.get('status'))
print('Primary hypothesis:', research.get('primary_hypothesis'))

In [ ]:
AUTO_UNASSIGN = True
RUN_RECENCY_RESEARCH = True
WRITE_FULL_BUNDLE = False  # Keep A100/Drive time low; set True for deep per-profile diagnostic zips.

import json
import pandas as pd
from pathlib import Path
from google.colab import runtime
from yenibot.experiment import (
    experiment_settings,
    prepare_training_holdout_split,
    profile_run_dir,
    run_experiment_matrix,
    run_recency_ensemble_research,
    write_experiment_diagnostics,
)

try:
    labeled_path = f'{DATA_DIR}/processed/labeled_1h.parquet'
    if not os.path.exists(labeled_path):
        raise FileNotFoundError(f'Missing labeled input; run notebooks 01-03 first: {labeled_path}')
    labeled_full = pd.read_parquet(labeled_path)
    assert labeled_full['timestamp'].is_monotonic_increasing
    assert not labeled_full['timestamp'].duplicated().any()
    holdout_path = f'{DATA_DIR}/processed/holdout_1h.parquet'
    labeled, holdout, holdout_meta = prepare_training_holdout_split(
        labeled_full,
        cfg,
        holdout_path=holdout_path,
    )
    cfg.setdefault('experiments', {})['holdout'] = holdout_meta
    min_selection_rows = (
        cfg['walk_forward']['train_bars']
        + cfg['walk_forward']['val_bars']
        + cfg['walk_forward']['test_bars']
        + cfg['walk_forward']['purge_bars']
        + cfg['walk_forward']['embargo_bars']
    )
    if len(labeled) <= min_selection_rows:
        raise ValueError(f'Not enough selection rows after holdout split: {len(labeled)}')

    settings = experiment_settings(cfg)
    print('Full labeled rows:', len(labeled_full))
    print('Selection/training rows:', len(labeled))
    print('Holdout rows:', len(holdout))
    print('Selection data end:', holdout_meta['selection_data_end'])
    print('Holdout data start:', holdout_meta['holdout_data_start'])
    print('Future OOS ready:', holdout_meta.get('future_oos_ready'))
    print('Future OOS next action:', holdout_meta.get('next_action'))
    print('Experiment mode:', settings.get('mode', 'staged'))
    print('Control profile:', settings.get('control_profile'))
    print('Candidate profiles:', settings.get('candidate_profiles', []))
    print('Always-full profiles:', settings.get('always_full_profiles', []))
    print('Seed audit:', settings.get('seed_audit', {}))

    train_result = run_experiment_matrix(
        labeled,
        cfg,
        checkpoint_dir=CHECKPT_DIR,
    )
    print('Experiment run:', train_result['run_id'])
    print('Experiment dir:', train_result['run_dir'])
    print('Training executed scopes:', train_result.get('training_executed_count'))
    print('Training reused scopes:', train_result.get('training_skipped_count'))

    recency_completed = False
    recency_cfg = research.get('recency_ensemble', {})
    if RUN_RECENCY_RESEARCH and recency_cfg.get('enabled', False):
        control_scope = profile_run_dir(
            CHECKPT_DIR,
            train_result['run_id'],
            settings['control_profile'],
        ) / 'full'
        recency_result = run_recency_ensemble_research(
            frame=labeled,
            scope_dir=control_scope,
            config=cfg,
            output_dir=train_result['run_dir'] / 'recency_research',
        )
        print('Recency ensemble research:', recency_result.get('status'))
        print('Fit operations in recency research: 0')
        recency_completed = recency_result.get('status') in {'completed', 'reused'}
        if not recency_result.get('summary', pd.DataFrame()).empty:
            display(recency_result['summary'])
        audit = recency_result.get('eligibility_audit', pd.DataFrame())
        if not audit.empty:
            assert audit['eligible'].all(), 'Recency eligibility audit failed'
    handoff_path = Path(CHECKPT_DIR) / 'notebook04_run.json'
    handoff_path.write_text(json.dumps({
        'run_id': train_result['run_id'],
        'repo_branch': repo_branch,
        'repo_commit': repo_commit,
        'recency_research_completed': recency_completed,
    }, indent=2), encoding='utf-8')
    print('Diagnostics run handoff:', handoff_path)

    diagnostics = write_experiment_diagnostics(
        checkpoint_dir=CHECKPT_DIR,
        config=cfg,
        output_dir=REPORT_DIR,
        run_id=train_result['run_id'],
        write_full_bundles=WRITE_FULL_BUNDLE,
    )
    display(diagnostics['comparison'])
    if not diagnostics.get('profile_blend').empty:
        display(diagnostics['profile_blend'])
    if not diagnostics.get('holdout_evaluation').empty:
        print('Holdout evaluation:')
        display(diagnostics['holdout_evaluation'])

    print('Full bundle enabled:', diagnostics.get('write_full_bundles'))
    print('Single download bundle:', diagnostics.get('bundle_zip'))
    print('Latest bundle shortcut:', diagnostics.get('latest_bundle_zip'))
    print('Slim summary bundle:', diagnostics.get('slim_bundle_zip'))
    print('Latest slim shortcut:', diagnostics.get('latest_slim_bundle_zip'))
    print('Auto review:', diagnostics.get('auto_review_path'))
    print('Next actions:', diagnostics.get('next_actions_path'))
    print('Decision:', diagnostics['decision'])
finally:
    if AUTO_UNASSIGN:
        runtime.unassign()
